# DA5453 Module 2 Coding Assignment — Training and testing a reward model

**Course:** Learning from Human Preferences  
**Instructor:** Suryanarayana Sankagiri

> **Provisional preview.** This notebook illustrates the expected structure, concepts,
> and approximate workload of the Module 2 coding assignment. The dataset, model specification, numerical
> results, and some questions will be revised before the final assignment is released.

In this assignment, you will train a small reward model from pairwise preference data.
You will then test the model, identify a shortcut that it has learnt, and repair it.

No previous knowledge of language models, text embeddings, or PyTorch is assumed. The
first part of the notebook introduces the coding tools that you will need. Run the cells
in order. Complete the cells marked **TODO** and write short observations where requested.

Keep the following files in the same folder as this notebook:

`train.csv`, `train_balanced.csv`, `eval_1.csv`, `eval_2.csv`, `eval_3.csv`,
and `answers.csv`.


## 1. Setup

Google Colab's free CPU runtime is sufficient. If you are using Colab, upload the six CSV
files through the file panel before running the data-loading cell.

The first command installs the text-embedding library. The package version and model
revision are fixed so that everybody uses the same encoder for this preview.


In [ ]:
%pip -q install sentence-transformers==5.1.2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer

torch.manual_seed(0)
np.random.seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


## 2. The data

Each preference example has three fields:

- `prompt`: the question given to the system;
- `chosen`: the response that is preferred;
- `rejected`: the less suitable response.

In this preview dataset, a response is preferred when it answers the prompt. The examples
are deliberately simple. This allows us to study the behaviour of the reward model without
requiring any background in large language models.


In [ ]:
def load_rows(path):
    # Read a CSV file and return one tuple for each row.
    frame = pd.read_csv(path)
    return [tuple(row) for row in frame.itertuples(index=False, name=None)]


train = load_rows("train.csv")
train_balanced = load_rows("train_balanced.csv")
eval_1 = load_rows("eval_1.csv")
eval_2 = load_rows("eval_2.csv")
eval_3 = load_rows("eval_3.csv")
probe = load_rows("answers.csv")

print(f"{len(train)} training pairs")
print(f"{len(eval_1)} pairs in each evaluation set")

prompt, chosen, rejected = train[0]
print("\nPROMPT:\n", prompt)
print("\nCHOSEN RESPONSE:\n", chosen)
print("\nREJECTED RESPONSE:\n", rejected)


## 3. A short introduction to text embeddings

A text-embedding model maps a piece of text to a vector:

\[
f(\text{text})\in\mathbb{R}^{d}.
\]

Texts with related meanings often have vectors pointing in similar directions. We will use
the frozen encoder `all-MiniLM-L6-v2`, for which \(d=384\). The encoder is not trained in
this assignment.

We normalise each embedding to have length one. The dot product of two such embeddings is
their cosine similarity:

\[
\operatorname{sim}(a,b)=f(a)^\top f(b).
\]

A large value suggests that the two texts are semantically related. This is not a measure
of factual correctness.


In [ ]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"

encoder = SentenceTransformer(
    MODEL_NAME,
    revision=MODEL_REVISION,
    device=device,
)


def embed(texts):
    # Map a list of texts to a matrix with one embedding per row.
    return encoder.encode(
        list(texts),
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )


example_prompt, example_answer, example_wrong_answer = probe[0]
E = embed([example_prompt, example_answer, example_wrong_answer])

print("embedding matrix shape:", E.shape)
print("prompt and correct answer:",
      round(float(E[0] @ E[1]), 3))
print("prompt and unrelated answer:",
      round(float(E[0] @ E[2]), 3))


### 3.1 A similarity heatmap

The following heatmap compares twelve prompts with their twelve correct responses. Prompt
\(i\) and response \(i\) belong together, so the diagonal entries are the matched pairs.
All code for this plot is provided.


In [ ]:
n_show = 12
show_prompts = [p for p, _, _ in probe[:n_show]]
show_answers = [a for _, a, _ in probe[:n_show]]

P_show = embed(show_prompts)
A_show = embed(show_answers)
similarities = P_show @ A_show.T

fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(similarities, cmap="viridis", vmin=0, vmax=1)
ax.set_xlabel("Correct response number")
ax.set_ylabel("Prompt number")
ax.set_xticks(range(n_show), range(1, n_show + 1))
ax.set_yticks(range(n_show), range(1, n_show + 1))
fig.colorbar(image, ax=ax, label="cosine similarity")
ax.set_title("Similarity between prompts and correct responses")
plt.tight_layout()
plt.show()


**Your observation.** In two or three sentences, describe the diagonal of the heatmap.
Are all matched prompt–response pairs more similar than all unmatched pairs?


_Write your observation here._


### 3.2 Looking at embeddings in two dimensions

The embeddings have 384 coordinates, so we cannot draw them directly. Principal component
analysis (PCA) finds a two-dimensional projection that retains as much variation as
possible. The plot below is only an illustration: distances can change when 384 dimensions
are compressed to two.

The PCA calculation is supplied. You are not required to implement it.


In [ ]:
def project_to_2d(X):
    # Return the first two principal-component coordinates.
    centred = X - X.mean(axis=0, keepdims=True)
    _, _, Vt = np.linalg.svd(centred, full_matrices=False)
    return centred @ Vt[:2].T


wrong_answers = [a for _, _, a in probe[:n_show]]
all_embeddings = np.vstack([
    embed(show_prompts),
    embed(show_answers),
    embed(wrong_answers),
])
points = project_to_2d(all_embeddings)

prompt_points = points[:n_show]
answer_points = points[n_show:2 * n_show]
wrong_points = points[2 * n_show:]

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(prompt_points[:, 0], prompt_points[:, 1],
           marker="o", s=65, label="prompts")
ax.scatter(answer_points[:, 0], answer_points[:, 1],
           marker="^", s=65, label="correct responses")
ax.scatter(wrong_points[:, 0], wrong_points[:, 1],
           marker="x", s=65, label="unrelated responses")

for i in range(n_show):
    ax.plot(
        [prompt_points[i, 0], answer_points[i, 0]],
        [prompt_points[i, 1], answer_points[i, 1]],
        color="grey",
        alpha=0.35,
        linewidth=1,
    )

ax.set_title("A two-dimensional PCA view of the embeddings")
ax.set_xlabel("first principal component")
ax.set_ylabel("second principal component")
ax.legend()
plt.tight_layout()
plt.show()


**Your observation.** What does the plot suggest about the embedding model? Mention one
reason why this two-dimensional picture should not be treated as a complete test of the
encoder.


_Write your observation here._


## 4. A small PyTorch example

PyTorch trains a model by repeatedly performing four operations:

```python
optimiser.zero_grad()
loss = ...
loss.backward()
optimiser.step()
```

The example below fits the line \(y=2x+1\). It is independent of the reward-model
assignment. Run it and read the comments carefully.


In [ ]:
# Five training examples. Each row is one example and each example has one feature.
x_toy = torch.tensor([[-2.0], [-1.0], [0.0], [1.0], [2.0]])
y_toy = 2 * x_toy + 1

# Linear(1, 1) represents y_hat = weight * x + bias.
toy_model = torch.nn.Linear(1, 1)
toy_optimiser = torch.optim.SGD(toy_model.parameters(), lr=0.05)
toy_losses = []

for step in range(120):
    toy_optimiser.zero_grad()                 # discard gradients from the previous step
    prediction = toy_model(x_toy)             # forward pass
    loss = ((prediction - y_toy) ** 2).mean() # one scalar measuring the error
    loss.backward()                           # compute gradients of the loss
    toy_optimiser.step()                      # update the weight and bias
    toy_losses.append(float(loss.detach()))

weight = float(toy_model.weight.detach().squeeze())
bias = float(toy_model.bias.detach().squeeze())
print("learnt weight:", round(weight, 3))
print("learnt bias:  ", round(bias, 3))

plt.figure(figsize=(6, 3.5))
plt.plot(toy_losses)
plt.yscale("log")
plt.xlabel("training step")
plt.ylabel("mean-squared error")
plt.title("The loss falls as the line is fitted")
plt.tight_layout()
plt.show()


The reward model below uses the same training pattern. The main differences are the form
of the loss and the fact that each example contains a pair of responses.


## 5. Reward modelling as Bradley–Terry learning

A reward model assigns a scalar score to a prompt–response pair. We begin with a simpler
response-only model:

\[
r(y)=w^\top f(y)+b.
\]

For an observed comparison \(y^+\succ y^-\), the Bradley–Terry model gives

\[
\Pr(y^+\succ y^-)
=
\sigma\bigl(r(y^+)-r(y^-)\bigr).
\]

The mean negative log-likelihood is

\[
L(w)
=
-\frac1n\sum_{i=1}^n
\log\sigma\bigl(r(y_i^+)-r(y_i^-)\bigr).
\]


### Part 1 — Train the baseline reward model

Complete the Bradley–Terry loss and the training loop. The remaining code records the
loss, pairwise accuracy, and mean reward margin during training.


In [ ]:
# Compute the frozen embeddings only once.
X_chosen = torch.tensor(
    embed([chosen for _, chosen, _ in train]),
    dtype=torch.float32,
)
X_rejected = torch.tensor(
    embed([rejected for _, _, rejected in train]),
    dtype=torch.float32,
)

reward_head = torch.nn.Linear(384, 1)


def reward(X):
    # Return one scalar reward for each row of X.
    return reward_head(X).squeeze(-1)


def bt_loss(Xc, Xr):
    # Mean Bradley–Terry negative log-likelihood.

    # TODO: compute r(chosen) - r(rejected), then return the mean negative
    # log-sigmoid of this margin.
    raise NotImplementedError


optimiser = torch.optim.Adam(reward_head.parameters(), lr=0.05)
history = {"step": [], "loss": [], "accuracy": [], "margin": []}

for step in range(401):

    # TODO: perform one PyTorch training step.
    pass


    if step % 10 == 0:
        with torch.no_grad():
            margins = reward(X_chosen) - reward(X_rejected)
            history["step"].append(step)
            history["loss"].append(float(bt_loss(X_chosen, X_rejected)))
            history["accuracy"].append(float((margins > 0).float().mean()))
            history["margin"].append(float(margins.mean()))

print("final training loss:", round(history["loss"][-1], 4))
print("training accuracy:  ", round(history["accuracy"][-1], 3))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

axes[0].plot(history["step"], history["loss"])
axes[0].set_title("Training loss")
axes[0].set_xlabel("step")

axes[1].plot(history["step"], history["accuracy"])
axes[1].set_ylim(0, 1.05)
axes[1].set_title("Training accuracy")
axes[1].set_xlabel("step")

axes[2].plot(history["step"], history["margin"])
axes[2].axhline(0, color="black", linewidth=1)
axes[2].set_title("Mean chosen–rejected margin")
axes[2].set_xlabel("step")

plt.tight_layout()
plt.show()


**Your observation.** Describe how the loss, accuracy, and mean margin change during
training. Does high training accuracy by itself show that the model has learnt the right
rule?


_Write your observation here._


### Part 2 — Evaluate the model

Pairwise accuracy is the fraction of examples for which the chosen response receives the
higher reward. Complete the function below. The supplied code then evaluates the model on
three datasets and draws two figures.


In [ ]:
@torch.no_grad()
def pairwise_margins(rows):
    chosen_embeddings = torch.tensor(
        embed([chosen for _, chosen, _ in rows]),
        dtype=torch.float32,
    )
    rejected_embeddings = torch.tensor(
        embed([rejected for _, _, rejected in rows]),
        dtype=torch.float32,
    )
    return reward(chosen_embeddings) - reward(rejected_embeddings)


@torch.no_grad()
def accuracy(rows):

    # TODO: return the fraction of chosen–rejected margins that are positive.
    raise NotImplementedError


evaluation_sets = {
    "eval_1": eval_1,
    "eval_2": eval_2,
    "eval_3": eval_3,
}
evaluation_accuracy = {
    name: accuracy(rows)
    for name, rows in evaluation_sets.items()
}

for name, value in evaluation_accuracy.items():
    print(name, "accuracy:", round(value, 3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(
    evaluation_accuracy.keys(),
    evaluation_accuracy.values(),
    color=["#4C78A8", "#F2CF5B", "#E45756"],
)
axes[0].axhline(0.5, color="black", linestyle="--", linewidth=1,
                label="random choice")
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("pairwise accuracy")
axes[0].set_title("Accuracy on the three evaluation sets")
axes[0].legend()

for name, rows in evaluation_sets.items():
    axes[1].hist(
        pairwise_margins(rows).numpy(),
        bins=12,
        alpha=0.45,
        label=name,
    )
axes[1].axvline(0, color="black", linewidth=1)
axes[1].set_xlabel("reward(chosen) − reward(rejected)")
axes[1].set_ylabel("number of pairs")
axes[1].set_title("Distribution of reward margins")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Inspect corresponding examples from the three evaluation sets.
for index in range(3):
    print(f"\nPAIR {index + 1}")
    for name, rows in evaluation_sets.items():
        prompt, chosen, rejected = rows[index]
        print(f"\n{name}")
        print(" chosen:  ", chosen)
        print(" rejected:", rejected)


**Your Part 2 answer.** State what changes across the three evaluation sets. Use the
figures and inspected examples to explain what the baseline reward model has learnt.
Explain why one accuracy is below \(0.5\).


_Write your answer here._


### Part 3 — Measure the shortcut directly

We now change one factor at a time.

- The **style gap** compares the same correct response with and without the enthusiastic
  wrapper.
- The **quality gap** compares a correct and an unrelated response written in the same
  plain style.

Both gaps are measured in the reward units of this model. Their absolute magnitudes have
no independent meaning, but they can be compared with each other.


In [ ]:
def add_style(text):
    return (
        "Great question, I really like this one! "
        + text
        + " Hope this helps, you've got this!"
    )


@torch.no_grad()
def score(responses):
    embeddings = torch.tensor(embed(responses), dtype=torch.float32)
    return reward(embeddings).numpy()


on_topic = [answer for _, answer, _ in probe]
off_topic = [answer for _, _, answer in probe]
styled_on_topic = [add_style(answer) for answer in on_topic]


# TODO: compute one style change and one quality change for every probe example.
style_changes = ...
quality_changes = ...


print("mean style gap:  ", round(float(np.mean(style_changes)), 3))
print("mean quality gap:", round(float(np.mean(quality_changes)), 3))

fig, ax = plt.subplots(figsize=(7, 4))
positions = [1, 2]
ax.boxplot([style_changes, quality_changes], positions=positions, widths=0.5)

rng = np.random.default_rng(0)
for position, values, colour in [
    (1, style_changes, "#E45756"),
    (2, quality_changes, "#4C78A8"),
]:
    jitter = rng.normal(0, 0.035, size=len(values))
    ax.scatter(
        np.full(len(values), position) + jitter,
        values,
        alpha=0.65,
        s=25,
        color=colour,
    )

ax.axhline(0, color="black", linewidth=1)
ax.set_xticks(positions, ["style gap", "quality gap"])
ax.set_ylabel("change in reward")
ax.set_title("What changes the baseline model's reward?")
plt.tight_layout()
plt.show()


**Your Part 3 answer.** Compare the two gap distributions. What is the baseline model
responding to?


_Write your answer here._


### Part 4 — Repair the reward model

There are two separate problems:

1. The original training data make style a reliable shortcut.
2. A response-only model cannot judge whether a response answers a particular prompt.

The file `train_balanced.csv` removes the first problem by balancing the stylistic pattern.
To address the second problem, the model must see a prompt–response interaction.

#### Why simple concatenation is insufficient

Suppose a linear head is applied to concatenated prompt and response embeddings:

\[
r(x,y)=w_x^\top f(x)+w_y^\top f(y)+b.
\]

Both responses in a comparison have the same prompt. Therefore,

\[
r(x,y^+)-r(x,y^-)
=
w_y^\top\bigl(f(y^+)-f(y^-)\bigr).
\]

The prompt term and bias cancel. A linear head on simple concatenation is therefore still
unable to judge whether a response matches its prompt.

We will add the elementwise interaction \(f(x)\odot f(y)\). A linear head can assign
weights to these interaction coordinates.


In [ ]:
def make_features(prompts, responses, mode):
    P = embed(prompts)
    Y = embed(responses)

    if mode == "response_only":
        return Y
    if mode == "concat":

        # TODO: place the prompt and response embeddings beside each other.
        raise NotImplementedError

    if mode == "interaction":

        # TODO: concatenate P, Y, and their elementwise product P * Y.
        raise NotImplementedError

    raise ValueError(f"unknown feature mode: {mode}")


def train_and_evaluate(train_rows, mode, steps=500, learning_rate=0.05):
    torch.manual_seed(0)

    Xc = torch.tensor(
        make_features(
            [p for p, _, _ in train_rows],
            [c for _, c, _ in train_rows],
            mode,
        ),
        dtype=torch.float32,
    )
    Xr = torch.tensor(
        make_features(
            [p for p, _, _ in train_rows],
            [r for _, _, r in train_rows],
            mode,
        ),
        dtype=torch.float32,
    )

    head = torch.nn.Linear(Xc.shape[1], 1)

    def local_reward(X):
        return head(X).squeeze(-1)

    local_optimiser = torch.optim.Adam(head.parameters(), lr=learning_rate)
    for _ in range(steps):
        local_optimiser.zero_grad()
        margin = local_reward(Xc) - local_reward(Xr)
        loss = -F.logsigmoid(margin).mean()
        loss.backward()
        local_optimiser.step()

    @torch.no_grad()
    def local_accuracy(rows):
        Ec = torch.tensor(
            make_features(
                [p for p, _, _ in rows],
                [c for _, c, _ in rows],
                mode,
            ),
            dtype=torch.float32,
        )
        Er = torch.tensor(
            make_features(
                [p for p, _, _ in rows],
                [r for _, _, r in rows],
                mode,
            ),
            dtype=torch.float32,
        )
        return float((local_reward(Ec) > local_reward(Er)).float().mean())

    return [local_accuracy(rows) for rows in [eval_1, eval_2, eval_3]]


configurations = [
    ("response only | original", train, "response_only"),
    ("response only | balanced", train_balanced, "response_only"),
    ("concat | original", train, "concat"),
    ("concat | balanced", train_balanced, "concat"),
    ("interaction | original", train, "interaction"),
    ("interaction | balanced", train_balanced, "interaction"),
]

result_names = []
result_values = []
for name, training_rows, mode in configurations:
    values = train_and_evaluate(training_rows, mode)
    result_names.append(name)
    result_values.append(values)
    print(f"{name:28s}", "  ".join(f"{value:.2f}" for value in values))

result_values = np.asarray(result_values)

fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(result_values, cmap="YlGn", vmin=0, vmax=1)
ax.set_xticks(range(3), ["eval_1", "eval_2", "eval_3"])
ax.set_yticks(range(len(result_names)), result_names)
ax.set_title("Pairwise accuracy after changing the data and features")

for row in range(result_values.shape[0]):
    for col in range(result_values.shape[1]):
        value = result_values[row, col]
        colour = "white" if value < 0.35 else "black"
        ax.text(col, row, f"{value:.2f}", ha="center", va="center", color=colour)

fig.colorbar(image, ax=ax, label="pairwise accuracy")
plt.tight_layout()
plt.show()


**Your Part 4 answer.** Identify a configuration that performs well on all three
evaluation sets. Explain, using the heatmap, why changing only the data is insufficient
and why changing only the features is insufficient.


_Write your answer here._


## 6. What to submit

For the final assignment, you will be asked to submit a completed notebook with all cells
run and all figures visible, together with a short written report. The exact submission
instructions and mark distribution will be given with the final version.

For this preview, you may run the notebook and inspect the tasks. No submission is
required.
